In [170]:
import numpy as np
import pandas as pd
from tqdm import tqdm

In [55]:
from nba_api.stats.endpoints import playercareerstats
from nba_api.stats.static import players
from nba_api.stats.endpoints import commonplayerinfo
from nba_api.stats.endpoints.commonallplayers import CommonAllPlayers
from nba_api.stats.endpoints import playergamelog

In [56]:
from __future__ import print_function
import pickle
import os.path
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

In [57]:
SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
spreadsheet_id = '16KJmOPm_2zAl2W1X2w265an-d7yMolOMfmITrTINsfI'
# sheet_range = 'Sheet1!B2:H'
creds = None
# The file token.pickle stores the user's access and refresh tokens, and is
# created automatically when the authorization flow completes for the first
# time.
if os.path.exists('token.pickle'):
    with open('token.pickle', 'rb') as token:
        creds = pickle.load(token)
# If there are no (valid) credentials available, let the user log in.
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            'credentials.json', SCOPES)
        creds = flow.run_local_server(port=0)
    # Save the credentials for the next run
    with open('token.pickle', 'wb') as token:
        pickle.dump(creds, token)
        
service = build('sheets', 'v4', credentials=creds)
sheet = service.spreadsheets()

In [58]:
# Get all players drafted

result = sheet.values().get(spreadsheetId=spreadsheet_id,
                            range='Draft!B2:H').execute()
drafted_players_by_team = result.get('values', [])

In [59]:
drafted_players_by_team = np.transpose(np.asarray(drafted_players_by_team))
drafted_players = drafted_players_by_team.flatten()

In [60]:
all_players = CommonAllPlayers().common_all_players.get_data_frame()

In [9]:
def get_player_id(player_list,common_all_players):
    player_array = []
    
    use_player_list = common_all_players.loc[common_all_players['DISPLAY_FIRST_LAST'].isin(player_list)]
    
    for index,row in use_player_list.iterrows():
        player_array.append({'player_name': row['DISPLAY_FIRST_LAST'],'player_id': row['PERSON_ID']})
        
    return player_array

In [63]:
player_dict = get_player_id(drafted_players,all_players)

In [64]:
julian_result = sheet.values().get(spreadsheetId=spreadsheet_id,
                            range='Julian!A2:A').execute()
julian_players = julian_result.get('values', [])


drew_result = sheet.values().get(spreadsheetId=spreadsheet_id,
                            range='Drew!A2:A').execute()
drew_players = drew_result.get('values', [])


jono_result = sheet.values().get(spreadsheetId=spreadsheet_id,
                            range='Jono!A2:A').execute()
jono_players = jono_result.get('values', [])


sam_result = sheet.values().get(spreadsheetId=spreadsheet_id,
                            range='Sam!A2:A').execute()
sam_players = sam_result.get('values', [])


joe_result = sheet.values().get(spreadsheetId=spreadsheet_id,
                            range='Joe!A2:A').execute()
joe_players = joe_result.get('values', [])


con_result = sheet.values().get(spreadsheetId=spreadsheet_id,
                            range='Con!A2:A').execute()
con_players = con_result.get('values', [])


ryan_result = sheet.values().get(spreadsheetId=spreadsheet_id,
                            range='Ryan!A2:A').execute()
ryan_players = ryan_result.get('values', [])

In [65]:
print(julian_players)
print(drew_players)
print(jono_players)
print(sam_players)
print(joe_players)
print(con_players)
print(ryan_players)

[['Damian Lillard'], ['Kevin Durant'], ['Russell Westbrook'], ['T.J. Warren'], ['DeMar DeRozan'], ['Buddy Hield'], ['Bojan Bogdanovic'], ['Aaron Gordon'], ['Evan Fournier'], ['Jordan Clarkson'], ["Devonte' Graham"], ['Duncan Robinson'], ['Kevin Love'], ['Jusuf Nurkic'], ['Will Barton'], ['JJ Redick']]
[['Giannis Antetokounmpo'], ['Karl-Anthony Towns'], ['Zach LaVine'], ['CJ McCollum'], ['Joel Embiid'], ["De'Aaron Fox"], ['John Collins'], ['Kelly Oubre Jr.'], ['Fred VanVleet'], ['LaMarcus Aldridge'], ['Lauri Markkanen'], ['Dillon Brooks'], ['Bogdan Bogdanovic'], ['Davis Bertans'], ['Marvin Bagley III'], ['Markelle Fultz']]
[['Luka Doncic'], ['LeBron James'], ['Nikola Jokic'], ['Collin Sexton'], ['Dennis Schroder'], ['Chris Paul'], ['Bam Adebayo'], ['Danilo Gallinari'], ['Tim Hardaway Jr.'], ['Joe Harris'], ['Darius Garland'], ['OG Anunoby'], ['Norman Powell'], ['Myles Turner'], ['Brandon Clarke'], ['Patty Mills']]
[['Trae Young'], ['Donovan Mitchell'], ['Shai Gilgeous-Alexander'], ['Pas

In [13]:
def player_points(player_name,player_dict):
    
    point_total = 0
    
    indiv_dict = [player for player in player_dict if player['player_name'] == player_name][0]
    indiv_id = indiv_dict['player_id']
    
    indiv_logs = playergamelog.PlayerGameLog(player_id = indiv_id,season=2020).get_data_frames()[0]
    
    point_total = indiv_logs['PTS'].sum()
    
    return point_total

In [19]:
player_points('Donovan Mitchell',player_dict)

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [232]:
# best way to do this is grab some cell (made in variable), get the player name there, get their points, and then put the points one cell to the right

#need to initialize spreadsheet_id and sheet first
#grabs player name, returns their point totals as a list
def update_player_points(sheet_name,cell):
    sheet_range = sheet_name + '!' + cell
    new_sheet_range = sheet_name +'!' + chr(ord(cell[0])+1)+cell[1:]
    player_result = sheet.values().get(spreadsheetId=spreadsheet_id,
                            range=sheet_range).execute()
    
    player_name = player_result.get('values',[])[0][0]
    
    new_points = player_points(player_name,player_dict)
    
    return [str(new_points)]
    

In [17]:
# batch get everything, append the lists, then batch push
def update_player_points(sheet_name,cell):
    sheet_range = sheet_name + '!' + cell
#     new_sheet_range = sheet_name +'!' + chr(ord(cell[0])+1)+cell[1:]
    
    
    player_result = sheet.values().get(spreadsheetId=spreadsheet_id,
                            range=sheet_range).execute()
    
    player_names = player_result.get('values',[])#[0][0]
    
    new_points = []
    
    for p in player_names:
        new_points.append([player_points(p[0],player_dict)])
        
    return new_points
    
    
#     return new_points 
    
#     return player_names
    
#     return player_names,new_points
    
#     new_points = player_points(player_name,player_dict)
    
#     return [str(new_points)]


In [18]:
update_player_points('Joe','A2:A17')

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [ ]:
values = []
    values = [[str(new_points)]]
    body = {
        'values': values
    }
    result = sheet.values().update(
    spreadsheetId=sheet_id, range=new_sheet_range,
    valueInputOption='USER_ENTERED', body=body).execute()

In [148]:
sheet_names = ['Julian','Drew','Jono','Sam','Joe','Con','Ryan']

In [235]:
for s in sheet_names:
    for c in cell_ranges:
        update_player_points(s,c)

HttpError: <HttpError 429 when requesting https://sheets.googleapis.com/v4/spreadsheets/16KJmOPm_2zAl2W1X2w265an-d7yMolOMfmITrTINsfI/values/Ryan%21A10?alt=json returned "Quota exceeded for quota metric 'Read requests' and limit 'Read requests per minute per user' of service 'sheets.googleapis.com' for consumer 'project_number:471295436627'.". Details: "[{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'RATE_LIMIT_EXCEEDED', 'domain': 'googleapis.com', 'metadata': {'quota_limit': 'ReadRequestsPerMinutePerUser', 'service': 'sheets.googleapis.com', 'consumer': 'projects/471295436627', 'quota_metric': 'sheets.googleapis.com/read_requests'}}]">

In [234]:
for c in cell_ranges:
    update_player_points('Julian',c)

In [242]:
values = [
    ['test'],['test4'],
    # Additional rows
]
data = [
    {
        'range': 'Sheet13!A1:A2',
        'values': values
    },
    # Additional ranges to update ...
]
body = {
    'valueInputOption': 'USER_ENTERED',
    'data': data
}
result = service.spreadsheets().values().batchUpdate(
    spreadsheetId=spreadsheet_id, body=body).execute()

In [166]:
from nba_api.stats.endpoints import leaguedashplayerstats

In [167]:
player_stats = leaguedashplayerstats.LeagueDashPlayerStats().get_data_frames()[0]

In [49]:
player_points('Zach LaVine',player_dict)

117

In [51]:
player_stats[['PLAYER_NAME','PTS']]

,PLAYER_NAME,PTS
0,Aaron Gordon,57
1,Aaron Holiday,36
2,Aaron Nesmith,8
3,Adam Mokoka,4
4,Al Horford,31
...,...,...
420,Willy Hernangomez,1
421,Yuta Watanabe,0
422,Zach LaVine,117
423,Zeke Nnaji,0


In [154]:
player_stats.query('PLAYER_NAME == "Kemba Walker"')

,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,MIN,...,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,CFID,CFPARAMS


In [168]:
# batch get everything, append the lists, then batch push
def update_player_points(sheet_name):
    sheet_range = sheet_name + '!' + 'A2:A17'
    new_sheet_range = sheet_name + '!' + 'B2:B17'
    
    player_result = sheet.values().get(spreadsheetId=spreadsheet_id,
                            range=sheet_range).execute()
    
    player_names = player_result.get('values',[])
    
    player_names = np.asarray(player_names)
    
    player_names = player_names.flatten()
    
    new_points = []
    
    
    for p in player_names:
        try:
            new_points.append([str(player_stats.query(f'PLAYER_NAME == "{p}"')['PTS'].iloc[0])])
        except IndexError:
            f'No points scored by {p}, adding a 0.'
            new_points.append([str(0)])

    values = new_points
    data = [
        {
            'range': new_sheet_range,
            'values': values
        },
        # Additional ranges to update ...
    ]
    body = {
        'valueInputOption': 'USER_ENTERED',
        'data': data
    }
    result = service.spreadsheets().values().batchUpdate(
        spreadsheetId=spreadsheet_id, body=body).execute()



In [165]:
update_player_points('Sam')

In [169]:
for s in sheet_names:
    update_player_points(s)

TimeoutError: [Errno 60] Operation timed out